In [ ]:
"""
Gemma4 26B A4B LoRA Fine-tuning (Unsloth)
RTX 6000 Pro Blackwell (96GB) / chi 환경
completion_only_loss 방식 (prompt-completion 데이터셋)
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
MODEL_ID = "unsloth/gemma-4-26B-A4B-it"
DATASET_PATH = "./data/7_qwen_dataset_train.jsonl"
OUTPUT_DIR = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split"

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 7e-5
LORA_RANK = 16
LORA_ALPHA = 32
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 모델 + 토크나이저
# -------------------------
print("📦 모델 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
# 또는 base_model 경유
print(model.base_model.config._attn_implementation)

# -------------------------
# LoRA
# -------------------------
print("🔧 LoRA 어댑터 부착...")
model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,
)
model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    """
    messages를 prompt(system+user)와 completion(assistant)으로 분리.
    prompt는 assistant turn 시작 마커까지 포함하고,
    completion은 순수 assistant 내용만.
    """
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    # prompt: system + user + "<|im_start|>assistant\n" 까지
    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,   # assistant 시작 마커까지 포함
    )

    # completion: assistant 본문 + "<|im_end|>\n"
    # (EOS가 반드시 학습되도록 명시)
    completion_text = asst_msgs[0]["content"] + _tok.eos_token


    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check: 첫 샘플 확인
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

# prompt가 assistant 마커로 끝나는지
assert "<|turn>model" in _s["prompt"][-150:], \
    f"❌ prompt가 model turn 마커를 포함하지 않음: {repr(_s['prompt'][-150:])}"
# completion EOS 체크는 모델의 실제 eos_token으로
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,   # ← 핵심: completion 부분만 loss
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

#trainer.train(resume_from_checkpoint=True)
trainer.train()

# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

✅ 컴파일 캐시 삭제 확인 완료
✅ UNSLOTH_TARGET_GB = 4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ unsloth TARGET_GB = 4
📦 모델 로딩...
==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.4.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.936 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: `flash_attention_2` is not supported for `gemma4` because max attention head dim 512 exceeds the Flash Attention 2 limit of 256 - defaulting to `sdpa`.


Loading weights: 100%|██████████| 1013/1013 [00:04<00:00, 219.31it/s]


sdpa
sdpa
🔧 LoRA 어댑터 부착...
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['experts.gate_up_proj', 'experts.down_proj']


...

Filter (num_proc=32): 100%|██████████| 63080/63080 [00:01<00:00, 48818.50 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ 필터: 63080개 → 63080개 (0개 제외)
🚀 학습 시작...
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=32): 100%|██████████| 63080/63080 [01:19<00:00, 794.09 examples/s] 
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 63,080 | Num Epochs = 1 | Total steps = 7,885
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 505,429,248 of 26,311,363,120 (1.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.963717
20,2.873015
30,3.163424


In [ ]:
# %% LoRA 어댑터 재로드 → merge → 표준 토크나이저로 저장
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

from unsloth import FastModel
from transformers import AutoTokenizer

BASE_MODEL     = "unsloth/gemma-4-26B-A4B-it"
LORA_DIR       = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split"
MERGED_DIR     = LORA_DIR + "-merged"
MAX_SEQ_LENGTH = 131072

print("📦 LoRA 어댑터 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print(f"🔀 merge 중... → {MERGED_DIR}")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

In [ ]:
"""
Qwen3.5-9B LoRA 이어서 훈련 (Unsloth)
기존 epoch 1 학습된 어댑터를 로드해서 추가 epoch 학습
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
PREV_LORA_DIR = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split"
OUTPUT_DIR    = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split-ep2"
DATASET_PATH  = "./data/7_qwen_dataset_train.jsonl"

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1          # 추가 1 epoch (누적 2 epoch 효과)
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 3e-5    # 기존 7e-5의 약 절반, 미세 조정용
LORA_RANK = 16
LORA_ALPHA = 32
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 기존 LoRA 어댑터를 붙인 모델 로드
# -------------------------
print(f"📦 기존 LoRA 어댑터 로딩: {PREV_LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=PREV_LORA_DIR,       # ← base가 아니라 기존 어댑터 경로
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
print(model.base_model.config._attn_implementation)

# 어댑터 이미 로드됨 → get_peft_model 호출하지 않음
# 그래디언트가 LoRA 파라미터에만 흐르는지 확인
model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + _tok.eos_token

    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

assert "<|turn>model" in _s["prompt"][-150:], \
    f"❌ prompt가 model turn 마커를 포함하지 않음: {repr(_s['prompt'][-150:])}"
# completion EOS 체크는 모델의 실제 eos_token으로
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 이어서 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,              # 0.05 → 0.01 (이미 데워진 상태)
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=43,                        # 기존과 다른 seed → 다른 순서로 데이터 노출
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

trainer.train()


# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

In [ ]:
# %% LoRA 어댑터 재로드 → merge → 표준 토크나이저로 저장
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

from unsloth import FastModel
from transformers import AutoTokenizer

BASE_MODEL     = "unsloth/gemma-4-26B-A4B-it"
LORA_DIR       = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split-ep2"
MERGED_DIR     = LORA_DIR + "-merged"
MAX_SEQ_LENGTH = 131072

print("📦 LoRA 어댑터 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print(f"🔀 merge 중... → {MERGED_DIR}")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

In [ ]:
"""
Qwen3.5-9B LoRA 이어서 훈련 - Epoch 3 (ep2 → ep3)
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
PREV_LORA_DIR = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split-ep2"
OUTPUT_DIR    = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split-ep3"
DATASET_PATH  = "./data/7_qwen_dataset_train.jsonl"

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1          # 추가 1 epoch (누적 3 epoch 효과)
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1.5e-5  # ep2의 3e-5 → 절반. 수렴 후반에 더 작게
LORA_RANK = 16
LORA_ALPHA = 32
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 기존 LoRA 어댑터 로드
# -------------------------
print(f"📦 기존 LoRA 어댑터 로딩: {PREV_LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=PREV_LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
print(model.base_model.config._attn_implementation)

model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + _tok.eos_token

    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)

# ep1(seed 42), ep2(seed 43)와 다른 순서로 섞기
dataset = dataset.shuffle(seed=44)

print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

assert "<|turn>model" in _s["prompt"][-150:], \
    f"❌ prompt가 model turn 마커를 포함하지 않음: {repr(_s['prompt'][-150:])}"
# completion EOS 체크는 모델의 실제 eos_token으로
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 ep3 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=44,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

trainer.train()


# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

In [ ]:
# %% LoRA 어댑터 재로드 → merge → 표준 토크나이저로 저장
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

from unsloth import FastModel
from transformers import AutoTokenizer

BASE_MODEL     = "unsloth/gemma-4-26B-A4B-it"
LORA_DIR       = "./model/gemma4-26b-a4b-lora-teloptextgen-rank64-split-ep3"
MERGED_DIR     = LORA_DIR + "-merged"
MAX_SEQ_LENGTH = 131072

print("📦 LoRA 어댑터 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print(f"🔀 merge 중... → {MERGED_DIR}")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")